# 01 — Pipeline-ul zilnic de știri financiare

**Echipa:** Tofan Bogdan, Manea Alina-Alexandra, Burcă Alina  
**Materia:** NLP

Acest notebook reproduce flow-ul `Daily news agent` din N8N ca proiect Python rulabil pe Google Colab.

## Ce face

1. Preia știri din **NewsData.io** (politică, economie, Bitcoin, Ethereum, ecosistem Solana) — 5 surse în paralel.
2. Face **scraping** pe linkul fiecărui articol și curăță HTML-ul.
3. Folosește un **Information Extractor (DeepSeek)** ca să extragă doar conținutul util.
4. Clasifică sentimentul cu **FinBERT** (HuggingFace Inference API).
5. Agregă metadatele și împarte articolele în **Bullish / Neutral / Bearish**.
6. Trei **sub-agenți DeepSeek** sintetizează fiecare categorie.
7. Un **agent principal Grok** combină cele 3 brief-uri și interoghează **Pinecone** pentru paralele istorice.
8. Raportul final + audio Opus (OpenAI TTS) sunt trimise pe **Telegram**.

Tot ce vezi mai jos este implementat în pachetul `src/` — celulele de aici doar orchestrează.

## 1. Setup

Pe Colab decomentează liniile de mai jos pentru a clona repo-ul și a instala dependențele. Apoi încarcă fișierul `.env` în panel-ul Files.

In [1]:
import shutil, os
shutil.rmtree('/content/financial-news-agent', ignore_errors=True)

In [2]:
%cd /content
!git clone -b Fin-news-agent-dev https://github.com/BogdanT54/financial-news-agent.git
%cd /content/financial-news-agent
!pip install -q -r requirements.txt

/content
Cloning into 'financial-news-agent'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 84 (delta 35), reused 76 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 43.64 KiB | 3.64 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/financial-news-agent
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 42.4 MB/

In [3]:
import sys, os
# Asigură-te că rădăcina proiectului e în sys.path când rulăm din notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, os.path.abspath(".."))

from src.config import get_settings
settings = get_settings()
print("Pinecone index :", settings.PINECONE_INDEX)
print("MongoDB        :", settings.MONGO_DB)
print("Sub-agent model:", settings.MODEL_SUB_AGENT)
print("Main model     :", settings.MODEL_MAIN_AGENT)
print("DRY_RUN        :", settings.DRY_RUN)

Pinecone index : fin-news-documents
MongoDB        : fin-news-db
Sub-agent model: deepseek/deepseek-v4-flash
Main model     : x-ai/grok-4.3
DRY_RUN        : True


## 2. Verificare credentials

Verifică rapid că ai completat cheile esențiale în `.env`.

In [4]:
required = {
    "NewsData.io": settings.NEWSDATA_API_KEY,
    "HuggingFace": settings.HF_API_TOKEN,
    "OpenRouter":  settings.OPENROUTER_API_KEY,
    "OpenAI":      settings.OPENAI_API_KEY,
    "Pinecone":    settings.PINECONE_API_KEY,
    "MongoDB":     settings.MONGO_URI,
    "Telegram":    settings.TELEGRAM_BOT_TOKEN,
}
for name, val in required.items():
    print(f"  {name:<12}: {'OK' if val else 'LIPSEȘTE'}")

  NewsData.io : OK
  HuggingFace : OK
  OpenRouter  : OK
  OpenAI      : OK
  Pinecone    : OK
  MongoDB     : OK
  Telegram    : OK


## 3. Pas cu pas: fetch știri

Cele 5 fetchere rulează în paralel cu `ThreadPoolExecutor`.

In [5]:
from src.news_fetcher import fetch_all_news, merge_news_sources

responses = fetch_all_news()
for source, payload in responses.items():
    n = len(payload.get("results") or [])
    print(f"  {source:<10}: {n} articole")

raw_articles = merge_news_sources(responses)
print(f"\nTotal după merge: {len(raw_articles)} articole")

  politics  : 10 articole
  economics : 10 articole
  bitcoin   : 10 articole
  ethereum  : 10 articole
  solana    : 10 articole

Total după merge: 50 articole


## 4. Pas cu pas: scraping + clean + extract pentru un articol

Demonstrăm întregul lanț pe un singur articol înainte să rulăm pe toate.

In [6]:
from src.scraper import fetch_article_html, clean_html, extract_main_content

sample = raw_articles[0]
print("Title:", sample.get("title"))
print("Link :", sample.get("link"))

raw_html = fetch_article_html(sample["link"])
print(f"\nHTML brut: {len(raw_html)} caractere")

clean = clean_html(raw_html)
print(f"După clean: {len(clean)} caractere")
print("\nPrimele 500 caractere:\n", clean[:500])

Title: They started IVF, then split. Now who gets custody of the embryos?
Link : https://www.bostonglobe.com/2026/05/24/nation/ivf-divorce-embryos-custody/

HTML brut: 865722 caractere
După clean: 9006 caractere

Primele 500 caractere:
 Skip to main content Sections Epaper Sports Red Sox Patriots Celtics Bruins High Schools Colleges Women's Sports TV & Radio Metro Weather Politics Transportation Education Cambridge & Somerville Around Mass. Investigations Obituaries Death Notices Games & Puzzles Align Crossword Mini Crossword KenKen Sudoku Word Flower Business Technology Economy Housing Real Estate Biotech The Fine Print Bold Types Politics Money, Power, Inequality Climate Spotlight Opinion Ideas Columns and Op-eds Editorials L


In [7]:
extracted = extract_main_content(clean)
print("Conținut extras (primele 800 caractere):\n")
print(extracted[:800])

Conținut extras (primele 800 caractere):

They started IVF, then split. Now who gets custody of the embryos?

Erin Millender in the kitchen of her Airbnb in Salt Lake City on May 17. KIM RAFF/NYT

NEW YORK -- More than anything else in the world, Erin Millender longed to be a mother. She already had a daycare picked out, a Pack 'n Play stashed in her basement. She'd tried Chinese pregnancy teas and midnight fertility ceremonies under a full moon in the Caribbean Sea. Whatever it took to have a child. Now in her mid-40s, Millender knew she was running out of time. She had already spent several years attempting in vitro fertilization, with no luck. She'd decided to give IVF one more try. "What's a good day to come in?" Millender asked when she called the clinic in July 2023, hoping to have an embryo placed inside her uterus within a


## 5. Pas cu pas: clasificare FinBERT

Apelăm modelul `ProsusAI/finbert` prin HuggingFace Inference API.

In [8]:
from src.finbert import classify_sentiment

predictions = classify_sentiment(
    sample.get("title", ""),
    sample.get("description", ""),
    extracted,
)
for p in predictions:
    print(f"  {p['label']:<10}: {p['score']:.4f}")

  neutral   : 0.9147
  negative  : 0.0517
  positive  : 0.0336


## 6. Pas cu pas: agregare metadate

Funcția `build_article_record` întoarce un dict cu exact aceleași câmpuri ca nodul `Aggregate data` din N8N.

In [9]:
from src.aggregator import build_article_record

record = build_article_record(sample, extracted, predictions)
for k in ["title", "sentiment", "confidence_level", "finbert_score",
         "score_positive", "score_neutral", "score_negative", "coins"]:
    print(f"  {k:<18}: {record[k]}")

  title             : They started IVF, then split. Now who gets custody of the embryos?
  sentiment         : Neutral
  confidence_level  : High
  finbert_score     : 0.914681
  score_positive    : 0.03363
  score_neutral     : 0.914681
  score_negative    : 0.051689
  coins             : []


## 7. Demo formattere (Bullish / Neutral / Bearish)

Folosim un dataset mic mock ca să vedem cum arată blocurile colorate.

In [10]:
from src.formatters import format_bullish_block

mock = [
    {**record, "sentiment": "Bullish", "finbert_label": "positive",
     "finbert_score": 0.92, "confidence_level": "High",
     "score_positive": 0.92, "score_neutral": 0.05, "score_negative": 0.03}
]
print(format_bullish_block(mock))

🟢 BULLISH ARTICLES (1 found)
════════════════════════════════════════
🟢 Article 1 — BULLISH (STRONG)

Title       : They started IVF, then split. Now who gets custody of the embryos?
Source      : The Boston Globe · caroline kitchener
Published   : 2026-05-25
Link        : https://www.bostonglobe.com/2026/05/24/nation/ivf-divorce-embryos-custody/

Description : The issue is just starting to seep into politics, amid a push from the Trump administration to make IVF more accessible.

Content     : They started IVF, then split. Now who gets custody of the embryos?  Erin Millender in the kitchen of her Airbnb in Salt Lake City on May 17. KIM RAFF/NYT  NEW YORK -- More than anything else in the world, Erin Millender longed to be a mother. She already had a daycare picked out, a Pack 'n Play stashed in her basement. She'd tried Chinese pregnancy teas and midnight fertility ceremonies under a full moon in the Caribbean Sea. Whatever it took to have a child. Now in her mid-40s, Millender knew s

## 8. Rulare pipeline complet (end-to-end)

`run_daily_pipeline()` execută tot lanțul. Cu `DRY_RUN=True` (default în `.env`):
- nu scrie în Pinecone / Mongo
- nu trimite mesaje pe Telegram
- afișează raportul + audio inline în notebook

Pentru demo rapid folosește `max_articles=3` (sau 5).

In [13]:
from src.pipeline import run_daily_pipeline

result = run_daily_pipeline(max_articles=40, persist=True, send_telegram=True)
print("\n=== STATS ===")
for k, v in result["stats"].items():
    print(f"  {k:<22}: {v}")

→ Fetching news from NewsData.io (5 sources in parallel)...
   50 articles fetched
   limited to 40 for this run
→ Processing each article (scrape → clean → extract → FinBERT)...


articles: 100%|██████████| 40/40 [17:55<00:00, 26.88s/it]


   40 articles successfully processed
→ Persisting to Pinecone and MongoDB...
   [1/40] Neutral  | They started IVF, then split. Now who gets custody of t
   [2/40] Neutral  | The future is shaped by people who show up
   [3/40] Bearish  | Cockroaches bug India’s Narendra Modi and his court
   [4/40] Bearish  | Travel industry worries after Trump administration reit
   [5/40] Neutral  | The Abraham Accords’ Evolution
   [6/40] Bearish  | Donald Trump Transfers Troops to Poland: The Rise of th
   [7/40] Bearish  | Trump news at a glance: President defends himself from 
   [8/40] Neutral  | Teen Takeovers Expose a Culture Running Out of Adult Su
   [9/40] Neutral  | Memorial Day ceremonies happening in the Valley
   [10/40] Neutral  | Bernie Sanders rallies progressives for Graham Platner 
   [11/40] Neutral  | Silicon Valley takes its AI pitch to Pope Leo
   [12/40] Neutral  | Amprius and Matternet Partner to Advance Drone Delivery
   [13/40] Neutral  | Rush Street Interactive’s SWOT an

/content/financial-news-agent/src/pipeline.py:163: RuntimeWarning: coroutine '_send_text_async' was never awaited
  print(f"   Telegram send failed: {exc}")


### Raportul final

In [14]:
from IPython.display import Markdown, Audio, display

display(Markdown(result["report"]))

📊 OVERVIEW
Markets today reflect a clear consolidation phase, with 21 neutral articles outweighing 13 bearish and 6 bullish pieces. Institutional infrastructure advances in crypto sit alongside persistent ETF outflows and geopolitical friction. The dominant narrative centers on a confidence divide: sophisticated capital continues accumulating while retail sentiment and leverage unwind, leaving price action range-bound.

🟢 *BULLISH (6 articles)*
- Privacy assets lead rotation as RAIL surges nearly 200% and Monero, Dash, Zcash outperform, reflecting demand for anonymity amid regulatory pressure.
- Bitcoin rebounds on US-Iran deal speculation, interpreted as a risk-on catalyst that could ease sanctions uncertainty.
- Goldman Sachs forecasts 24-fold growth in AI token consumption by 2030, supporting long-term infrastructure demand despite near-term chip constraints.
- Ethereum ETF outflows of $216 million viewed as temporary rebalancing given cumulative inflows above $11 billion and stable net asset ratios.

🔴 *BEARISH (13 articles)*
- Bitcoin trend model turns negative per 10x Research, with Strategy’s potential sale of BTC holdings cited as trigger for $2.7 billion ETF outflows since 7 May.
- Spot buying pressure hits 2025 lows while Binance sees ten straight days of net BTC inflows, adding supply-side pressure.
- Over $127 million in derivatives liquidations in 24 hours, predominantly long positions on BTC and ETH, signals leveraged positioning caught offside.
- Geopolitical risks escalate via US troop shifts from Germany, sanctuary-city airport threats, and Republican divisions over any Iran deal, elevating risk premia.

⚪️ *NEUTRAL (21 articles)*
- SEC approves Nasdaq PHLX Bitcoin index options; Morgan Stanley files revised Solana ETF application.
- Bitcoin trades above $76,500 but faces resistance at $77,450; Altcoin Season Index remains at 33.
- Institutional accumulation by Strategy contrasts with retail caution amid sticky inflation and elevated Treasury yields.
- Vatican engagement by tech firms ahead of Pope Leo XIV’s AI encyclical and niche sector growth forecasts (action cameras, drone batteries) add structural context without directional bias.

🪙 SOLANA SPOTLIGHT
Morgan Stanley filed a revised Solana ETF application under the proposed MSOL ticker, signaling continued traditional-finance infrastructure build-out. No SOL price action, DeFi metrics, or protocol-specific developments appear in the dataset.

🕰️ HISTORICAL PARALLELS
No strong historical parallels found in the vector database.

⚡️ SIGNAL OF THE DAY
Bitcoin’s apparent demand has reached a 2025 low while ETF outflows accelerate and Strategy signals potential selling. This institutional inflection, layered on geopolitical uncertainty, points to a market where conviction is eroding faster than price, raising the probability of a decisive break below $76,000 in the near term.

──────────────────────────────────
📅 *Report Date:* 25 May 2026, 15:47
🤖 *Models used:* FinBERT + DeepSeek V4 Flash + Grok 4.3
🗄️ *Memory:* MongoDB + Pinecone Vector Store
──────────────────────────────────

### Audio raport (OpenAI TTS Opus)

In [15]:
if result["audio_bytes"]:
    display(Audio(data=result["audio_bytes"], autoplay=False))
else:
    print("Audio neindisponibil (TTS a eșuat sau e dezactivat).")

## 9. Mod live (Telegram + persistență)

Pentru o rulare reală (mesaj pe Telegram + scriere în Pinecone/Mongo), schimbă în `.env`:

```
DRY_RUN=false
```

Sau forțează din cod cu argumentele funcției:

In [ ]:
# Rulare LIVE — scrie în Pinecone+Mongo și trimite pe Telegram
# result_live = run_daily_pipeline(max_articles=10, dry_run=False)